# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：C:\Users\83895\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\data\E Commerce Dataset.xlsx
项目输出目录：C:\Users\83895\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [2]:
# 在此写下你的答案：
# 1. 每条记录代表平台上单个在售淘宝商品的观测样本，完整记录了该商品的ID、价格、销量、服务权益、属性参数等采集到的全部信息。
# 2. 本项目的目标变量是商品销量列，整个项目的清洗、特征构造工作都围绕销量相关分析展开，是本次研究的核心因变量。
# 3. CustomerID是用于区分不同客户的名义型身份编码，数字本身没有连续数值的数学含义：编号的大小、差值不代表客户的强弱、高低差异，对它做均值计算、连续建模这类数值分析都无法产出可解释的业务结论，仅能作为分组、去重的标识使用，因此不能当作普通连续数值参与后续分析。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [3]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    report = pd.DataFrame({
        "数据类型": data.dtypes,
        "缺失数量": data.isna().sum(),
        "缺失比例(%)": (data.isna().mean() * 100).round(2),
        "唯一值数量": data.nunique()
    })
    return report

# TODO：生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,str,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,str,0,0.00,7
Gender,str,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [4]:
# TODO：完成项目初始审计
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：", raw_df["CustomerID"].duplicated().sum())
print(raw_df["Churn"].value_counts())
print("流失率：", f"{(raw_df['Churn'].mean() * 100).round(2)}%")

for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 16.84%

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [5]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [6]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO: 复制数据，避免覆盖原始数据
    cleaned_df = data.copy()

    # TODO: 创建日志列表 logs
    logs = []

    # TODO: 删除完全重复行，并记录日志
    before_dup = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates()
    after_dup = len(cleaned_df)
    logs.append({
        "处理步骤": "删除完全重复行",
        "处理规则": "完全相同的记录不增加信息，存在则删除",
        "处理前记录数": before_dup,
        "处理后记录数": after_dup,
        "影响记录数": before_dup - after_dup
    })

    # TODO: 对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    for col in NUMERIC_MISSING_COLS:
        missing_before = cleaned_df[col].isna().sum()
        col_median = cleaned_df[col].median()
        cleaned_df[col] = cleaned_df[col].fillna(col_median)
        missing_after = cleaned_df[col].isna().sum()
        logs.append({
            "处理步骤": f"数值缺失填充 - {col}",
            "处理规则": "使用总体中位数填补，稳健且不将缺失误解为0",
            "处理前记录数": missing_before,
            "处理后记录数": missing_after,
            "影响记录数": missing_before - missing_after
        })

    # TODO: 对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    for col, mapping in CATEGORY_MAPPINGS.items():
        for old_val, new_val in mapping.items():
            affected = (cleaned_df[col] == old_val).sum()
            cleaned_df[col] = cleaned_df[col].replace(old_val, new_val)
            logs.append({
                "处理步骤": f"类别标准化 - {col}: {old_val} → {new_val}",
                "处理规则": "同一业务类别统一命名",
                "处理前记录数": affected,
                "处理后记录数": 0,
                "影响记录数": affected
            })

    # TODO: 将 Churn 和 Complain 转为整数类型
    for col in ["Churn", "Complain"]:
        total = len(cleaned_df)
        cleaned_df[col] = cleaned_df[col].astype(int)
        logs.append({
            "处理步骤": f"数据类型转换 - {col}",
            "处理规则": "转换为整数类型，适配后续建模分析",
            "处理前记录数": total,
            "处理后记录数": total,
            "影响记录数": total
        })

    # TODO: 返回 cleaned_df 与 cleaning_log
    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [7]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,完全相同的记录不增加信息，存在则删除,5630,5630,0
1,数值缺失填充 - Tenure,使用总体中位数填补，稳健且不将缺失误解为0,264,0,264
2,数值缺失填充 - WarehouseToHome,使用总体中位数填补，稳健且不将缺失误解为0,251,0,251
3,数值缺失填充 - HourSpendOnApp,使用总体中位数填补，稳健且不将缺失误解为0,255,0,255
4,数值缺失填充 - OrderAmountHikeFromlastYear,使用总体中位数填补，稳健且不将缺失误解为0,265,0,265
5,数值缺失填充 - CouponUsed,使用总体中位数填补，稳健且不将缺失误解为0,256,0,256
6,数值缺失填充 - OrderCount,使用总体中位数填补，稳健且不将缺失误解为0,258,0,258
7,数值缺失填充 - DaySinceLastOrder,使用总体中位数填补，稳健且不将缺失误解为0,307,0,307
8,类别标准化 - PreferredLoginDevice: Phone → Mobile P...,同一业务类别统一命名,1231,0,1231
9,类别标准化 - PreferredPaymentMode: COD → Cash on De...,同一业务类别统一命名,365,0,365


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [8]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO: 构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
tenure_bins = [0, 6, 12, 24, np.inf]
tenure_labels = ["0-6个月", "6-12个月", "12-24个月", "24个月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels)
# 兜底：分箱后如果仍有NaN，强制填充为"0-6个月"（最小区间，业务兜底）
cleaned_df["TenureGroup"] = cleaned_df["TenureGroup"].fillna("0-6个月")

# TODO: 新建 IsMobileLogin，移动端为 1，其他设备为 0
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)
# TODO: 生成 outlier_report（每行对应一个待检查字段）
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_list = []
for col in outlier_cols:
    summary = iqr_outlier_summary(cleaned_df[col])
    summary["字段名"] = col
    outlier_list.append(summary)

outlier_report = pd.DataFrame(outlier_list).set_index("字段名")
display(outlier_report)

,Q1,Q3,下限,上限,候选异常值数量
字段名,,,,,
WarehouseToHome,9.00,20.00,-7.50,36.50,2
OrderCount,1.00,3.00,-2.00,6.00,703
CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [9]:
# TODO：完成业务规则检查
business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长小于 0",
        "仓库距离小于 0",
        "订单数小于或等于 0",
        "返现金额小于 0"
    ],
    "不合规记录数": [
        (cleaned_df["Tenure"] < 0).sum(),
        (cleaned_df["WarehouseToHome"] < 0).sum(),
        (cleaned_df["OrderCount"] <= 0).sum(),
        (cleaned_df["CashbackAmount"] < 0).sum()
    ]
})
display(business_rule_report)

,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [10]:
# TODO：完成最终验收
quality_after = build_quality_report(cleaned_df)

# 断言校验：数值字段无缺失、类别已统一、衍生字段已生成
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍存在缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "登录设备类别未完成统一"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式类别未完成统一"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式类别未完成统一"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "衍生字段缺失"

# TODO: 导出下列文件，使用 utf-8-sig 编码
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

# TODO: 输出 outlier_report 和 business_rule_report
print("=== 候选异常值报告 ===")
display(outlier_report)
print("\n=== 业务规则检查报告 ===")
display(business_rule_report)

# TODO: 输出交付文件的路径
print("\n=== 交付文件路径 ===")
print(f"清洗前质量报告：{(OUTPUT_DIR / 'data_quality_before.csv').resolve()}")
print(f"清洗后质量报告：{(OUTPUT_DIR / 'data_quality_after.csv').resolve()}")
print(f"清洗流程日志：{(OUTPUT_DIR / 'cleaning_log.csv').resolve()}")
print(f"清洗后数据集：{(OUTPUT_DIR / 'ecommerce_customer_cleaned.csv').resolve()}")

=== 候选异常值报告 ===


,Q1,Q3,下限,上限,候选异常值数量
字段名,,,,,
WarehouseToHome,9.00,20.00,-7.50,36.50,2
OrderCount,1.00,3.00,-2.00,6.00,703
CashbackAmount,145.77,196.39,69.84,272.33,438



=== 业务规则检查报告 ===


,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0



=== 交付文件路径 ===
清洗前质量报告：C:\Users\83895\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\data_quality_before.csv
清洗后质量报告：C:\Users\83895\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\data_quality_after.csv
清洗流程日志：C:\Users\83895\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\cleaning_log.csv
清洗后数据集：C:\Users\83895\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

## GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] `output/day04_project/`包含清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [ ] 原始Excel没有被覆盖；
- [ ] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 已提交并推送到个人GitHub仓库。